# Test Distribution Analysis
**Goal:** Analyze test set feature distribution and identify potential domain shift.

## Questions to Answer
1. How do test features compare to training features?
2. Is there covariate shift between train and test?
3. Which features show the largest distribution differences?
4. Can we infer test sideslip distribution from feature patterns?

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.band_mapping import load_band_mapping, classify_band
from utils.reproducibility_utils import seed_everything

SEED = 42
seed_everything(SEED)

TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
TEST_PATH = PROJECT_ROOT / "datasets_challenge" / "test"

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")

## 1. Load Train and Test Data

In [ ]:
# Load training data
train_files = sorted(glob.glob(str(TRAIN_PATH / "train_part_*.csv")))
train_dfs = [pd.read_csv(f) for f in train_files]
train_data = pd.concat(train_dfs, ignore_index=True)
print(f"Train samples: {len(train_data):,}")

# Load test data
test_files = sorted(glob.glob(str(TEST_PATH / "test_part_*.csv")))
test_dfs = [pd.read_csv(f) for f in test_files]
test_data = pd.concat(test_dfs, ignore_index=True)
print(f"Test samples: {len(test_data):,}")

# Common columns (excluding sideslip which is only in train)
common_cols = [col for col in test_data.columns if col in train_data.columns and col != 'sideSlip']
print(f"\nCommon features: {len(common_cols)}")
print(common_cols)

## 2. Basic Statistics Comparison

In [ ]:
# Calculate statistics for train and test
feature_cols = ['handwheelAngle', 'throttle', 'brake', 'vxCG', 'axCG', 'ayCG', 
                'yawRate', 'rollRate', 'pitchRate', 'rollAngle', 'pitchAngle']

stats_comparison = []

for col in feature_cols:
    if col in train_data.columns and col in test_data.columns:
        train_stats = train_data[col].describe()
        test_stats = test_data[col].describe()
        
        stats_comparison.append({
            'Feature': col,
            'Train Mean': train_stats['mean'],
            'Test Mean': test_stats['mean'],
            'Mean Diff': test_stats['mean'] - train_stats['mean'],
            'Train Std': train_stats['std'],
            'Test Std': test_stats['std'],
            'Std Ratio': test_stats['std'] / train_stats['std'] if train_stats['std'] > 0 else 1,
        })

stats_df = pd.DataFrame(stats_comparison)
print("Feature Statistics Comparison:")
print(stats_df.to_string(index=False))

In [ ]:
# Visualize mean differences
fig = go.Figure(go.Bar(
    x=stats_df['Feature'],
    y=stats_df['Mean Diff'],
    marker_color=['red' if abs(x) > stats_df['Train Std'].values[i] * 0.5 else 'green' 
                  for i, x in enumerate(stats_df['Mean Diff'])],
    text=[f"{x:.3f}" for x in stats_df['Mean Diff']],
    textposition='outside'
))
fig.update_layout(
    title='Mean Difference (Test - Train)',
    xaxis_title='Feature',
    yaxis_title='Mean Difference',
    height=400,
    xaxis_tickangle=-45
)
fig.show()

## 3. Distribution Comparison (KDE Plots)

In [ ]:
# Sample for visualization
n_sample = min(50000, len(train_data), len(test_data))
train_sample = train_data.sample(n_sample, random_state=SEED)
test_sample = test_data.sample(n_sample, random_state=SEED)

# Key features to compare
key_features = ['vxCG', 'yawRate', 'ayCG', 'handwheelAngle', 'axCG', 'rollRate']

fig = make_subplots(rows=2, cols=3, subplot_titles=key_features)

for i, col in enumerate(key_features):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    
    # Train histogram
    fig.add_trace(
        go.Histogram(x=train_sample[col], nbinsx=100, name='Train', 
                     marker_color='blue', opacity=0.6, showlegend=(i==0)),
        row=row, col=col_idx
    )
    # Test histogram
    fig.add_trace(
        go.Histogram(x=test_sample[col], nbinsx=100, name='Test',
                     marker_color='red', opacity=0.6, showlegend=(i==0)),
        row=row, col=col_idx
    )

fig.update_layout(height=600, title='Feature Distribution: Train vs Test', barmode='overlay')
fig.show()

## 4. Statistical Tests for Distribution Shift

In [ ]:
# Kolmogorov-Smirnov test for each feature
ks_results = []

for col in feature_cols:
    if col in train_data.columns and col in test_data.columns:
        # Sample for KS test (it's slow on full data)
        train_vals = train_data[col].dropna().sample(min(10000, len(train_data)), random_state=SEED)
        test_vals = test_data[col].dropna().sample(min(10000, len(test_data)), random_state=SEED)
        
        # KS test
        ks_stat, p_value = stats.ks_2samp(train_vals, test_vals)
        
        ks_results.append({
            'Feature': col,
            'KS Statistic': ks_stat,
            'P-Value': p_value,
            'Significant Shift': 'YES' if p_value < 0.01 else 'NO'
        })

ks_df = pd.DataFrame(ks_results).sort_values('KS Statistic', ascending=False)
print("Kolmogorov-Smirnov Test Results:")
print(ks_df.to_string(index=False))
print("\n* KS Statistic > 0.1 with p < 0.01 indicates significant distribution shift")

In [ ]:
# Visualize KS statistics
fig = go.Figure(go.Bar(
    x=ks_df['Feature'],
    y=ks_df['KS Statistic'],
    marker_color=['red' if x > 0.1 else 'orange' if x > 0.05 else 'green' 
                  for x in ks_df['KS Statistic']],
    text=[f"{x:.3f}" for x in ks_df['KS Statistic']],
    textposition='outside'
))
fig.add_hline(y=0.1, line_dash='dash', line_color='red', annotation_text='Significant threshold')
fig.update_layout(
    title='KS Statistic by Feature (Higher = More Distribution Shift)',
    xaxis_title='Feature',
    yaxis_title='KS Statistic',
    height=400,
    xaxis_tickangle=-45
)
fig.show()

## 5. Velocity (vxCG) Distribution Analysis

In [ ]:
# vxCG is especially important due to threshold filtering
fig = make_subplots(rows=1, cols=2, subplot_titles=['Full Range', 'Above Threshold (>3.0)'])

# Full range
fig.add_trace(go.Histogram(x=train_data['vxCG'], nbinsx=100, name='Train', 
                           marker_color='blue', opacity=0.6), row=1, col=1)
fig.add_trace(go.Histogram(x=test_data['vxCG'], nbinsx=100, name='Test',
                           marker_color='red', opacity=0.6), row=1, col=1)

# Above threshold
train_above = train_data[train_data['vxCG'] > 3.0]['vxCG']
test_above = test_data[test_data['vxCG'] > 3.0]['vxCG']
fig.add_trace(go.Histogram(x=train_above, nbinsx=100, name='Train (>3)', 
                           marker_color='blue', opacity=0.6, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=test_above, nbinsx=100, name='Test (>3)',
                           marker_color='red', opacity=0.6, showlegend=False), row=1, col=2)

fig.add_vline(x=3.0, line_dash='dash', line_color='green', row=1, col=1)
fig.update_layout(height=400, title='Velocity Distribution (vxCG)', barmode='overlay')
fig.show()

# Statistics
print(f"Samples above threshold (vxCG > 3.0):")
print(f"  Train: {len(train_above):,} ({100*len(train_above)/len(train_data):.1f}%)")
print(f"  Test:  {len(test_above):,} ({100*len(test_above)/len(test_data):.1f}%)")

## 6. Infer Potential Sideslip Distribution in Test

In [ ]:
# Use correlated features to estimate test sideslip distribution
# Train a simple model to predict sideslip band from features

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Prepare training data
band_dict = load_band_mapping()
train_filtered = train_data[train_data['vxCG'] > 3.0].copy()
train_filtered['band'] = classify_band(train_filtered['sideSlip'].values, band_dict)

X_train = train_filtered[feature_cols].fillna(0)
y_train = train_filtered['band']

# Train classifier
clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=SEED, n_jobs=-1)
scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')
print(f"Band prediction accuracy (from features): {scores.mean():.3f} ± {scores.std():.3f}")

# Fit on full train and predict test
clf.fit(X_train, y_train)

In [ ]:
# Predict band distribution for test set
test_filtered = test_data[test_data['vxCG'] > 3.0].copy()
X_test = test_filtered[feature_cols].fillna(0)

if len(X_test) > 0:
    test_predicted_bands = clf.predict(X_test)
    
    # Compare distributions
    train_band_dist = y_train.value_counts(normalize=True) * 100
    test_band_dist = pd.Series(test_predicted_bands).value_counts(normalize=True) * 100
    
    print("Estimated Band Distribution (from features):")
    print(f"{'Band':<10} {'Train':<10} {'Test (Est)':<10}")
    print("-" * 30)
    for band in ['many', 'medium', 'few']:
        train_pct = train_band_dist.get(band, 0)
        test_pct = test_band_dist.get(band, 0)
        print(f"{band:<10} {train_pct:.1f}%{'':<5} {test_pct:.1f}%")

In [ ]:
# Visualize predicted band distribution
fig = go.Figure()

fig.add_trace(go.Bar(
    x=['many', 'medium', 'few'],
    y=[train_band_dist.get('many', 0), train_band_dist.get('medium', 0), train_band_dist.get('few', 0)],
    name='Train (Actual)',
    marker_color='blue'
))

if len(X_test) > 0:
    fig.add_trace(go.Bar(
        x=['many', 'medium', 'few'],
        y=[test_band_dist.get('many', 0), test_band_dist.get('medium', 0), test_band_dist.get('few', 0)],
        name='Test (Estimated)',
        marker_color='red'
    ))

fig.update_layout(
    title='Band Distribution: Train (Actual) vs Test (Estimated from Features)',
    xaxis_title='Band',
    yaxis_title='Percentage',
    barmode='group',
    height=400
)
fig.show()

## 7. Per-File Test Analysis

In [ ]:
# Analyze test files individually
test_file_stats = []

for f in test_files:
    df = pd.read_csv(f)
    filename = Path(f).name
    
    # Filter valid samples
    valid_df = df[df['vxCG'] > 3.0]
    
    stats = {
        'file': filename,
        'total_rows': len(df),
        'valid_rows': len(valid_df),
        'valid_pct': 100 * len(valid_df) / len(df) if len(df) > 0 else 0,
        'yawRate_std': valid_df['yawRate'].std() if len(valid_df) > 0 else 0,
        'ayCG_std': valid_df['ayCG'].std() if len(valid_df) > 0 else 0,
        'vxCG_mean': valid_df['vxCG'].mean() if len(valid_df) > 0 else 0,
    }
    
    # Predict bands for this file
    if len(valid_df) > 0:
        X_file = valid_df[feature_cols].fillna(0)
        predicted = clf.predict(X_file)
        stats['est_few_pct'] = 100 * (predicted == 'few').sum() / len(predicted)
        stats['est_medium_pct'] = 100 * (predicted == 'medium').sum() / len(predicted)
    else:
        stats['est_few_pct'] = 0
        stats['est_medium_pct'] = 0
    
    test_file_stats.append(stats)

test_file_df = pd.DataFrame(test_file_stats)
print("Test File Analysis:")
print(test_file_df[['file', 'valid_rows', 'est_few_pct', 'est_medium_pct']].to_string(index=False))

## 8. Feature Importance for Domain Shift

In [ ]:
# Which features are most different between train and test?
# Use simple domain classifier approach

# Create domain labels
train_sample_domain = train_data[feature_cols].sample(min(20000, len(train_data)), random_state=SEED).fillna(0)
test_sample_domain = test_data[feature_cols].sample(min(20000, len(test_data)), random_state=SEED).fillna(0)

X_domain = pd.concat([train_sample_domain, test_sample_domain])
y_domain = ['train'] * len(train_sample_domain) + ['test'] * len(test_sample_domain)

# Train domain classifier
domain_clf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=SEED, n_jobs=-1)
domain_scores = cross_val_score(domain_clf, X_domain, y_domain, cv=5, scoring='accuracy')
print(f"Domain classification accuracy: {domain_scores.mean():.3f}")
print("(0.5 = no shift, 1.0 = complete shift)")

# Feature importances
domain_clf.fit(X_domain, y_domain)
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': domain_clf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeatures Most Responsible for Domain Shift:")
print(importances.head(10).to_string(index=False))

In [ ]:
# Visualize feature importance for domain shift
fig = go.Figure(go.Bar(
    x=importances['Importance'],
    y=importances['Feature'],
    orientation='h',
    marker_color=['red' if x > 0.15 else 'orange' if x > 0.1 else 'green' 
                  for x in importances['Importance']]
))
fig.update_layout(
    title='Feature Importance for Train/Test Domain Shift',
    xaxis_title='Importance',
    yaxis_title='Feature',
    height=500
)
fig.show()

## Summary

### Domain Shift Analysis
| Metric | Value | Interpretation |
|--------|-------|----------------|
| Domain classifier accuracy | See above | Higher = more shift |
| Most shifted features | See above | Watch these in preprocessing |
| Estimated test few-shot % | See above | Compare to training |

### Key Findings
1. **Velocity filtering**: Similar thresholding behavior in train/test
2. **Feature distributions**: Some features show significant shift
3. **Estimated test distribution**: May differ from training
4. **Recommendations**:
   - Use stratified split that matches estimated test distribution
   - Pay attention to features with high domain shift
   - Consider domain adaptation techniques if shift is large

### Implications for Model Training
- If test has different distribution, validation metrics may be misleading
- Random stratified split (V3) may better match test if test is shuffled
- Temporal split (V2) may better match if test follows training temporally